In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head(20)

In [ ]:
df.info()
df.describe()
# df.shape

Drop Columns

In [ ]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)
# or
# df = df.drop(columns=["id", "Unnamed: 32"])

Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

X = df.iloc[:, 1:]
y = df.iloc[:, 0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X

In [ ]:
y

Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

In [ ]:
y_test
y_test.shape

Numpy to PyTorch Tensors

In [ ]:
import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
X_train_tensor
# X_train_tensor.shape

In [ ]:
y_test_tensor
y_test_tensor.shape

Building Model Architecture:

In [ ]:
class MyNN():
    def __init__(self, X_train_tensor):
        self.weights = torch.rand(X_train_tensor.shape[1], 1, dtype=torch.float32, requires_grad=True)
        self.bias = torch.zeros(1, dtype=torch.float32, requires_grad=True)

    def forward(self, X):
        z = torch.matmul(X_train_tensor, self.weights) + self.bias 
        # converts to 0-1
        y_pred_tensor = torch.sigmoid(z) 
        return y_pred_tensor

    def loss_function(self, y_pred_tensor, y_test_tensor):
        epsilon = 1e-7
        y_pred_tensor = torch.clamp(y_pred_tensor, epsilon, 1-epsilon)
        # Binary Cross-Entropy Loss!
        loss = -(y_train_tensor * torch.log(y_pred_tensor) + (1 - y_train_tensor) * torch.log(1 - y_pred_tensor)).mean()
        return loss


Initialize the Parameters:

In [ ]:
lr = 1e-4
epochs = 50

Training pipeline loop:

In [ ]:
model = MyNN(X_train_tensor)

for epoch in range(epochs):
    # forward pass!
    y_pred_tensor = model.forward(X_train_tensor)
    # loss calculate!
    loss = model.loss_function(y_pred_tensor, y_train_tensor)
    # backward pass!
    loss.backward()
    # parameters update!
    with torch.no_grad():
        model.weights -= lr*model.weights.grad
        model.bias -= lr*model.bias.grad
    # zero gradients!
    model.weights.grad.zero_()
    model.bias.grad.zero_()
    # print log!
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Evaluation:

In [ ]:
with torch.no_grad():
    y_pred_tensor = model.forward(X_train_tensor)

    # y_pred_tensor = (y_pred_tensor>0.5).float()
    # or
    for i in range(len(y_pred_tensor)):
        if y_pred_tensor[i] > 0.5:
            y_pred_tensor[i] = 1
        else:
            y_pred_tensor[i] = 0

    # accuracy = (y_pred_tensor == y_train_tensor).float().mean()
    # or
    correct = 0
    for i in range(len(y_pred_tensor)):
        if y_pred_tensor[i] == y_train_tensor[i]:
            correct = correct + 1
    accuracy = correct / len(y_pred_tensor)

    print(f'Accuracy: {accuracy}')